# M1 L1: AI-Native Data Engineering 101

**The entry point of the AI-Native DE course.  Plain English first, then the runnable proof.**

Duration: 60 min.  Topic: traditional DE vs AI-native DE; the 7 things AI-native DE owns.

Run the cells top to bottom. No API keys needed.

## Plain English (the 12-year-old version)

A **traditional data engineer** is like a librarian. Books come in, the librarian sorts them, puts them on the right shelves, and tells you where to find a book when you ask.

An **AI-native data engineer** is like a librarian who also owns the **search engine**. The librarian doesn't just shelve the book — they decide **how the search engine reads it**, **what version of the reader is installed**, **whether the reader is biased**, **how much each search costs**, and **whether the search accidentally shows you a book you're not allowed to read**.

Same librarian. Three times the metadata. Twice the things that can break.

## The 7 things AI-native DE owns

1. **Model identity** — every embedding has a `model_id` and version
2. **Embedding drift** — measuring how much a model upgrade changes the space
3. **Prompt versioning** — the template that produced this answer
4. **Eval set lineage** — the golden set that scored this answer
5. **Content hash** — the dedupe key, not the URL
6. **ACL tags at ingest** — so query-time filter is fast
7. **Cost / latency budget** — per-token $ and per-call P95

Without these 7, you have a chatbot. With them, you have a system.

In [1]:
# === Standard setup ===
import sys
from pathlib import Path

HERE = Path.cwd()
def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return HERE

ROOT = find_root(HERE)
sys.path.insert(0, str(ROOT))

# Load the lesson by file path (filename starts with a digit, can't import)
import importlib.util
_p = ROOT / 'lessons' / 'm01-foundations' / 'L1-ai-native-de-101.py'
_spec = importlib.util.spec_from_file_location('L1', _p)
_L1 = importlib.util.module_from_spec(_spec)
sys.modules['L1'] = _L1
_spec.loader.exec_module(_L1)

print(f'Setup OK. Lesson module: {_L1.LESSON_NUMBER}')
print(f'Root: {ROOT}')

Setup OK. Lesson module: M1.L1
Root: /Users/vishnoiprem/PycharmProjects/data-etl-ml/ai-course-public/course/ai-native-de


In [2]:
# Run the 10 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L1, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 10 tests...

  PASS  test_ai_native_pipeline_count_and_fields
  PASS  test_content_hash_stable
  PASS  test_cosine_identical
  PASS  test_cosine_orthogonal
  PASS  test_embedding_drift_identical
  PASS  test_l2_norm_known
  PASS  test_l2_norm_zero
  PASS  test_llm_cost_simple
  PASS  test_llm_cost_zero
  PASS  test_traditional_pipeline_count

10/10 tests pass, 0 fail


In [3]:
# === Section 1: traditional vs AI-native pipeline ===
trad = _L1.traditional_pipeline(_L1.SAMPLE_DOCS)
ai = _L1.ai_native_pipeline(_L1.SAMPLE_DOCS)
from dataclasses import asdict
print(f'Traditional: {len(trad)} records, {len(asdict(trad[0]))} fields')
print(f'AI-native  : {len(ai)} records, {len(asdict(ai[0]))} fields')
print()
print('AI-native extra fields:')
trad_keys = set(asdict(trad[0]).keys())
for k in asdict(ai[0]).keys():
    if k not in trad_keys:
        print(f'  + {k}')

Traditional: 10 records, 4 fields
AI-native  : 10 records, 12 fields

AI-native extra fields:
  + model_id
  + model_version
  + embedding
  + embedding_norm
  + prompt_template_version
  + eval_set_id
  + content_hash
  + acl_tags


In [4]:
# === Section 2: embedding drift demo ===
import numpy as np
v1 = _L1.fake_embed(_L1.SAMPLE_DOCS[0]['text'])
v2 = v1 + np.random.default_rng(seed=99).random(_L1.EMBEDDING_DIM).astype(np.float32) * 0.5
d = _L1.embedding_drift(v1, v2)
print(f'cosine(v1, v2) = {d["cosine"]:.3f}   (1.0 = identical direction)')
print(f'L2(v1, v2)     = {d["l2"]:.3f}')
print()
print('Rule of thumb (assumption, not measured):')
print('  cosine >= 0.9  -> safe to roll out')
print('  0.7 <= cos < 0.9 -> re-rank golden set, may need re-tuning')
print('  cosine < 0.7    -> meaningful shift, full re-evaluation')

cosine(v1, v2) = 0.945   (1.0 = identical direction)
L2(v1, v2)     = 0.848

Rule of thumb (assumption, not measured):
  cosine >= 0.9  -> safe to roll out
  0.7 <= cos < 0.9 -> re-rank golden set, may need re-tuning
  cosine < 0.7    -> meaningful shift, full re-evaluation


In [5]:
# === Section 3: cost + latency for one LLM call ===
p, c = 1200, 250
cost = _L1.llm_cost(p, c)
lat = _L1.llm_latency(p, c)
print(f'One LLM call:')
print(f'  prompt_tokens     = {p}')
print(f'  completion_tokens = {c}')
print(f'  cost              = ${cost:.5f}')
print(f'  latency (P50 est) = {lat:.2f}s')
print()
for daily in [1_000, 10_000, 100_000, 1_000_000]:
    monthly = cost * daily * 30
    print(f'  {daily:>9,} calls/day  ->  ${monthly:>10,.2f}/mo')

One LLM call:
  prompt_tokens     = 1200
  completion_tokens = 250
  cost              = $0.00022
  latency (P50 est) = 11.00s

      1,000 calls/day  ->  $      6.52/mo
     10,000 calls/day  ->  $     65.25/mo
    100,000 calls/day  ->  $    652.50/mo
  1,000,000 calls/day  ->  $  6,525.00/mo


In [6]:
# === Section 4: a real retrieval over the 10 docs ===
query = 'how much vacation do I get per year'
qv = _L1.fake_embed(query)
scored = [(_L1.cosine_sim(qv, np.asarray(d.embedding, dtype=np.float32)), d) for d in ai]
scored.sort(key=lambda x: x[0], reverse=True)
print(f'Query: {query!r}\n')
print(f'{"rank":<5} {"cosine":>7} {"doc_id":<6} text')
print('-' * 70)
for i, (s, d) in enumerate(scored[:5], 1):
    print(f'{i:<5} {s:>7.3f} {d.doc_id:<6} {d.text[:55]!r}')

Query: 'how much vacation do I get per year'

rank   cosine doc_id text
----------------------------------------------------------------------
1       0.861 d04    'Expense policy: submit within 30 days, manager approval'
2       0.858 d02    '401k match: 100% of the first 3% of salary, 50% of the '
3       0.825 d09    'Incident response: P1 within 15 min, P2 within 1 hour, '
4       0.784 d08    'Data retention: customer PII 7 years, financial records'
5       0.763 d10    'Quarterly business review: revenue, gross margin, NPS, '


## Architecture (Mermaid)

```mermaid
flowchart LR
    A[Sources<br/>PDF / HTML / API] --> B[Ingest<br/>parse + chunk + embed]
    B --> C[(Indexes<br/>BM25 + Vector Store<br/>+ Metadata DB)]
    C --> D[Retrieval<br/>BM25 + dense + RRF + ACL]
    D --> E[Generation<br/>prompt + LLM + citations + abstain]
    E --> F[Observability<br/>traces + metrics + cost]
    F --> G[Eval<br/>golden set + regression gate]
    G -.feedback.-> D
```

**Read as:** Sources feed an ingestion pipeline that produces an index. Queries hit the index, get retrieved chunks, get filtered by ACL, get assembled into a prompt, get sent to an LLM, get observed, and get graded against a golden set — which then feeds back to tune retrieval.

## Reflect (10 min)

1. In your own words, why does a model version field belong in the metadata of every record, not just in a config file?
2. Pick one of the 7 things and explain how you'd implement it in a system you've worked on. Be specific about the failure mode if you didn't have it.
3. The lesson says 1M calls/day = ~$650/mo. What's the first thing you'd cut if the budget was $200/mo?
4. Re-run the retrieval demo with a different query. Did the top result change? Why or why not?
5. (Stretch) Replace `fake_embed` with a real sentence-transformers call. How does the ranking change?

---

**Read next:**
- `lessons/m02-vector-math/L2-vector-math-knn.ipynb` — the math behind `cosine_sim` and `l2_norm`, with a from-scratch KNN implementation
- `lessons/m03-embeddings/L3-embeddings-and-models.ipynb` — real embeddings, model selection, dual-index migration
- `SYLLABUS.md` — the 18 modules and 21-day plan